In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------
RANDOM_STATE = 42
# ---------------------------------------------------------
# Predictor and target definitions from Stage 1
# ---------------------------------------------------------
TARGET = "Machine failure"
NUMERIC_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
CATEGORICAL_FEATURES = [
    "Type",
]
FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
)
# ---------------------------------------------------------
# Locate project dataset
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ai4i2020.csv"
)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: "
        f"{DATA_PATH.resolve()}"
    )
# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
df = pd.read_csv(DATA_PATH)
print(
    f"Dataset loaded: "
    f"{df.shape[0]:,} rows × "
    f"{df.shape[1]} columns"
)

Dataset loaded: 10,000 rows × 14 columns


In [ ]:
# ---------------------------------------------------------
# Verify modelling variables
# ---------------------------------------------------------
required_columns = FEATURES + [TARGET]
missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]
if missing_columns:
    raise ValueError(
        f"Required columns are missing: "
        f"{missing_columns}"
    )

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values detected in modelling variables."
    )

# ---------------------------------------------------------
# Define predictors and target
# ---------------------------------------------------------
X = df[FEATURES].copy()
y = df[TARGET].copy()

# ---------------------------------------------------------
# Reproduce Stage 2 train/test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Development/training set: {len(X_train):,}")
print(f"Reserved final test set:  {len(X_test):,}")

print(
    f"\nTraining failure rate: "
    f"{100 * y_train.mean():.2f}%"
)

print(
    f"Reserved test failure rate: "
    f"{100 * y_test.mean():.2f}%"
)

print(
    "\nThe final test set will not be used "
    "for model selection in Stage 3."
)

Development/training set: 8,000
Reserved final test set:  2,000

Training failure rate: 3.39%
Reserved test failure rate: 3.40%

The final test set will not be used for model selection in Stage 3.


In [3]:
# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


# ---------------------------------------------------------
# Linear Regression model
# ---------------------------------------------------------

linear_model = LinearRegression()


# ---------------------------------------------------------
# Complete modelling pipeline
# ---------------------------------------------------------

linear_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", linear_model),
    ]
)


# ---------------------------------------------------------
# Stratified cross-validation
# ---------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Linear Regression pipeline created.")
print("Cross-validation: 5-fold stratified")

Linear Regression pipeline created.
Cross-validation: 5-fold stratified


In [4]:
# ---------------------------------------------------------
# Generate out-of-fold predictions
# ---------------------------------------------------------
# Every prediction is produced by a model that did not
# train on that particular observation.

failure_score_oof = cross_val_predict(
    linear_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict",
)


# ---------------------------------------------------------
# Regression performance
# ---------------------------------------------------------

regression_metrics = pd.DataFrame({
    "Score": {
        "MAE": mean_absolute_error(
            y_train,
            failure_score_oof,
        ),
        "RMSE": root_mean_squared_error(
            y_train,
            failure_score_oof,
        ),
        "R²": r2_score(
            y_train,
            failure_score_oof,
        ),
    }
})

print("Regression performance:")
display(regression_metrics.round(4))


# ---------------------------------------------------------
# Examine failure-score range
# ---------------------------------------------------------

minimum_score = failure_score_oof.min()
maximum_score = failure_score_oof.max()

below_zero = np.sum(
    failure_score_oof < 0
)

above_one = np.sum(
    failure_score_oof > 1
)

print(
    f"\nPredicted failure-score range: "
    f"{minimum_score:.4f} to "
    f"{maximum_score:.4f}"
)

print(
    f"Scores below 0: {below_zero:,}"
)

print(
    f"Scores above 1: {above_one:,}"
)


# ---------------------------------------------------------
# Convert failure score to class prediction
# ---------------------------------------------------------

CLASSIFICATION_THRESHOLD = 0.5

y_pred_oof = (
    failure_score_oof
    >= CLASSIFICATION_THRESHOLD
).astype(int)


# ---------------------------------------------------------
# Classification performance
# ---------------------------------------------------------

classification_metrics = pd.DataFrame({
    "Score": {
        "Accuracy": accuracy_score(
            y_train,
            y_pred_oof,
        ),
        "Balanced Accuracy":
            balanced_accuracy_score(
                y_train,
                y_pred_oof,
            ),
        "Precision": precision_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
    }
})

print("\nClassification performance:")
display(
    classification_metrics.round(4)
)


# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    y_train,
    y_pred_oof,
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual normal",
        "Actual failure",
    ],
    columns=[
        "Predicted normal",
        "Predicted failure",
    ],
)

print("\nCross-validated confusion matrix:")
display(cm_df)

Regression performance:


,Score
MAE,0.0774
RMSE,0.1692
R²,0.1253



Predicted failure-score range: -0.1397 to 0.3526
Scores below 0: 2,571
Scores above 1: 0

Classification performance:


,Score
Accuracy,0.9661
Balanced Accuracy,0.5000
Precision,0.0000
Recall,0.0000
F1-score,0.0000



Cross-validated confusion matrix:


,Predicted normal,Predicted failure
Actual normal,7729,0
Actual failure,271,0
